# files

> Files and cells over the gateway's contents and cells APIs

In [ ]:
#| default_exp files

[rustygate](https://github.com/AnswerDotAI/rustygate) serves two REST families beside the kernels API: a jupyter-inspired files API (`/api/contents`) and a cells API (`/api/cells`, per-cell operations on notebooks). This page builds their clients: `JupyAsyncFilesClient` addresses files and directories by path, `JupyAsyncCellsClient` binds to one notebook's cells, and `apply_ops` is the reference applier for the `cell_ops` change broadcasts that arrive on a kernel client's merged stream (`get_jmsg`, `channel` `'cells'`).

In [ ]:
#| export
import httpx2
from base64 import b64encode, b64decode
from fastcore.basics import patch, patch_to
from jupyasyncclient.core import KernelApi

In [ ]:
import asyncio, json, tempfile
from pathlib import Path
from queue import Empty
from fastcore.test import test_eq
from rustygate.tools import start_gateway
from jupyasyncclient.core import JupyAsyncKernelClient

## The files client

All three classes here share `KernelApi`'s HTTP plumbing: base URL, token, and the transport (a fresh client per request, or one you pass in). `session_id` is the author identity the gateway uses for echo suppression: a mutation carrying a kernel client's `session_id` is not broadcast back to that client's websocket. Left as `None`, writes carry no author and everyone subscribed hears them. A conditional write (`expected_hash=`) that loses the race raises `HashMismatch`, whose `hash` is the server's current file hash, which is what a retry needs.

In [ ]:
#| export
class HashMismatch(Exception):
    "A conditional write failed; `hash` is the server's current file hash."
    def __init__(self, hash):
        super().__init__(f'expected_hash is stale; the current hash is {hash}')
        self.hash = hash

class JupyAsyncFilesClient(KernelApi):
    "Files and directories over the gateway's contents API."
    def __init__(self, base_url, token=None, session_id=None, headers=None, timeout=30, http_client=None, verify=True):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        self.session_id = session_id

Every call goes through one helper. `_creq` treats a plain path as a contents-API path (an absolute `/api/...` route passes through untouched), appends the author and conditional-write query parameters, and converts a 409 into `HashMismatch`. `get`, `put`, `post`, and `patch` are its verb forms: `get` maps `kwargs` to query parameters, the others to the JSON body, so `get(path)` alone is the bare model and `put(path, type='directory')` is a whole request.


In [ ]:
#| export
@patch
async def _creq(self:JupyAsyncFilesClient, method, path, expected_hash=None, params=None, **kw):
    if not path.startswith('/'): path = f'/api/contents/{path}' if path else '/api/contents'
    p = dict(params or {}, session_id=self.session_id, expected_hash=expected_hash)
    try: return await self._request(method, path, params={k:v for k,v in p.items() if v is not None}, **kw)
    except httpx2.HTTPStatusError as e:
        if e.response.status_code==409: raise HashMismatch(e.response.json()['hash']) from e
        raise

@patch
async def get(self:JupyAsyncFilesClient, path='', **kwargs):
    "The model at `path`; `kwargs` become query parameters, e.g. `fields`."
    return await self._creq('GET', path, params=kwargs)

@patch
async def put(self:JupyAsyncFilesClient, path, expected_hash=None, **kwargs):
    "PUT with `kwargs` as the JSON body."
    return await self._creq('PUT', path, expected_hash=expected_hash, json=kwargs)

@patch
async def post(self:JupyAsyncFilesClient, path, expected_hash=None, **kwargs):
    "POST with `kwargs` as the JSON body."
    return await self._creq('POST', path, expected_hash=expected_hash, json=kwargs)

@patch_to(JupyAsyncFilesClient)
async def patch(self, path, /, expected_hash=None, **kwargs):
    "PATCH with `kwargs` as the JSON body; `path` is positional-only, freeing the name for the body."
    return await self._creq('PATCH', path, expected_hash=expected_hash, json=kwargs)


In [ ]:
#| export
@patch
async def write(self:JupyAsyncFilesClient, path, content, expected_hash=None):
    "Write `content` (`str` as text, `bytes` as base64), returning the model with its new `hash`."
    c,f = (b64encode(content).decode(),'base64') if isinstance(content, bytes) else (content,'text')
    return await self.put(path, expected_hash=expected_hash, content=c, format=f)

@patch
async def read(self:JupyAsyncFilesClient, path):
    "A file's contents: `str` for text, `bytes` for binary."
    m = await self.get(path, fields='content')
    return b64decode(m['content']) if m['format']=='base64' else m['content']

@patch
async def listing(self:JupyAsyncFilesClient, path='', fields=None):
    "The entries of directory `path`; `fields='hash'` adds each file's hash."
    return (await self.get(path, fields=fields))['content']


A live gateway to demonstrate against: the rustygate binary, serving a scratch directory as its files root.

In [ ]:
root = Path(tempfile.mkdtemp())
g = start_gateway(('rustygate', '--root', root))
fc = JupyAsyncFilesClient(g.url)
m = await fc.write('notes.txt', 'hello')
m

The same hash appears at every layer: the write's returned model, and the directory listing with `fields='hash'`.

In [ ]:
test_eq(await fc.read('notes.txt'), 'hello')
entry = next(e for e in await fc.listing(fields='hash') if e['name']=='notes.txt')
test_eq(entry['hash'], m['hash'])
await fc.get('notes.txt')

A stale `expected_hash` refuses the write and hands back the current hash. The retry needs no extra round trip:

In [ ]:
try: await fc.write('notes.txt', 'clobber', expected_hash='0'*64)
except HashMismatch as e: cur = e.hash
test_eq(cur, m['hash'])
m2 = await fc.write('notes.txt', 'hello2', expected_hash=cur)
assert m2['hash'] != cur
await fc.read('notes.txt')

Bytes round-trip through base64 without the caller seeing it:

In [ ]:
raw = bytes(range(256))
await fc.write('blob.bin', raw)
back = await fc.read('blob.bin')
test_eq(back, raw)
len(back)

In [ ]:
#| export
@patch
async def mkdir(self:JupyAsyncFilesClient, path):
    "Create directory `path`; parents are not created."
    return await self.put(path, type='directory')

@patch
async def rename(self:JupyAsyncFilesClient, path, to):
    "Rename `path` to `to`, returning the new model."
    return await self.patch(path, path=to)

@patch
async def copy(self:JupyAsyncFilesClient, src, to):
    "Copy `src` to `to`, returning the new model."
    return await self.post(to, copy_from=src)

@patch
async def delete(self:JupyAsyncFilesClient, path, expected_hash=None):
    "Delete a file or an empty directory."
    return await self._creq('DELETE', path, expected_hash=expected_hash)


In [ ]:
await fc.mkdir('sub')
await fc.copy('notes.txt', 'sub/notes.txt')
await fc.rename('sub/notes.txt', 'sub/renamed.txt')
test_eq([e['name'] for e in await fc.listing('sub')], ['renamed.txt'])
await fc.delete('sub/renamed.txt')
await fc.delete('sub')
await fc.delete('blob.bin')
[e['name'] for e in await fc.listing()]

## The cells client

`JupyAsyncCellsClient` binds one notebook path in its constructor, and its cells methods need no path argument. The inherited file methods still take explicit paths, handy for a dialog's sibling assets. It keeps a `hash` cursor, updated by every response that carries the file hash, so a conditional `apply` needs no bookkeeping at the call site. `client[id]` awaits one cell; a tuple of ids awaits a list.

In [ ]:
#| export
class JupyAsyncCellsClient(JupyAsyncFilesClient):
    "One notebook's cells over the gateway's cells API."
    def __init__(self, base_url, path, token=None, session_id=None, headers=None, timeout=30, http_client=None, verify=True):
        super().__init__(base_url, token=token, session_id=session_id, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        self.path,self.hash = path,None

    def __getitem__(self, ids): return self._lookup(ids)

In [ ]:
#| export
@patch
def _npath(self:JupyAsyncCellsClient): return f'/api/cells/{self.path}'

@patch
async def cells(self:JupyAsyncCellsClient, ids=None):
    "The notebook's cells in document order, optionally filtered to `ids`."
    if ids is not None and not isinstance(ids, str): ids = ','.join(ids)
    m = await self.get(self._npath(), ids=ids)
    self.hash = m['hash']
    return m['cells']

@patch
async def hashes(self:JupyAsyncCellsClient):
    "Per-cell `{'id','hash'}` rows: the cheap form for sync."
    m = await self.get(self._npath(), fields='hashes')
    self.hash = m['hash']
    return m['cells']

@patch
async def apply(self:JupyAsyncCellsClient, ops, conditional=False):
    "Apply `ops` atomically, returning ids of added cells; `conditional=True` sends the cursor as `expected_hash`."
    m = await self.post(self._npath(), expected_hash=self.hash if conditional else None, ops=ops)
    self.hash = m['hash']
    return m['added_ids']

@patch
async def _lookup(self:JupyAsyncCellsClient, ids):
    one = isinstance(ids, str)
    want = [ids] if one else list(ids)
    got = await self.cells(ids=want)
    if len(got)!=len(want): raise KeyError(', '.join(i for i in want if i not in {c['id'] for c in got}))
    return got[0] if one else got

A notebook to work on, written through the files client (any nbformat producer works: the cells API reads the file fresh per request):

In [ ]:
cells = [dict(id='aaa1', cell_type='code', source='1+1', metadata={}, outputs=[], execution_count=None),
    dict(id='bbb2', cell_type='markdown', source='# hi', metadata={})]
await fc.write('d.ipynb', json.dumps(dict(nbformat=4, nbformat_minor=5, metadata={}, cells=cells)))
nb = JupyAsyncCellsClient(g.url, 'd.ipynb')
[c['id'] for c in await nb.cells()]

Cell lookup by id, fastlite-style; a missing id raises `KeyError`:

In [ ]:
c = await nb['bbb2']
test_eq(c['source'], '# hi')
pair = await nb['aaa1','bbb2']
test_eq([c['id'] for c in pair], ['aaa1','bbb2'])
try: await nb['nope']
except KeyError as e: err = str(e)
err

An op batch applies atomically: a sparse `add` is normalized server-side and its generated id comes back in order, an `update` replaces the named keys wholesale, and any failure rolls the whole batch back.

In [ ]:
added = await nb.apply([
    dict(op='add', cell=dict(cell_type='code', source='2+2'), after='aaa1'),
    dict(op='update', id='bbb2', source='# hello'),
])
new_id, = added
[c['id'] for c in await nb.cells()]

The cursor makes conditional writes one keyword. After any fetch the cursor is current and `conditional=True` applies cleanly. A stale cursor raises `HashMismatch`, and assigning `e.hash` back is the whole retry:

In [ ]:
await nb.hashes()
await nb.apply([dict(op='delete', id=new_id)], conditional=True)
nb.hash = 'stale'
try: await nb.apply([dict(op='update', id='aaa1', source='6*7')], conditional=True)
except HashMismatch as e: nb.hash = e.hash
await nb.apply([dict(op='update', id='aaa1', source='6*7')], conditional=True)
(await nb['aaa1'])['source']

## Applying ops

`apply_ops` is the client-side twin of the server's applier: the same vocabulary, applied to a plain list of cell dicts. It is how a client keeps a local view current from change broadcasts without refetching. File-level ops (`rename`, `deleted`, `reset`) are the caller's business and raise here. A consumer that forgets to handle them hears about it.

In [ ]:
#| export
def apply_ops(cells, ops):
    "Apply a `cell_ops` list to `cells` in place, in order, returning it."
    for o in ops:
        ids = [c['id'] for c in cells]
        if o['op']=='add':
            at = ids.index(o['after'])+1 if o.get('after') else ids.index(o['before']) if o.get('before') else len(cells)
            cells.insert(at, o['cell'])
        elif o['op']=='update': cells[ids.index(o['id'])].update({k:v for k,v in o.items() if k not in ('op','id')})
        elif o['op']=='delete': del cells[ids.index(o['id'])]
        else: raise ValueError(f"unhandled op: {o['op']}")
    return cells

Applying the ops we just sent to a stale local view reproduces the server's result exactly:

In [ ]:
view = await nb.cells()
ops = [dict(op='add', cell=dict(id='ccc3', cell_type='code', source='3', metadata={}, outputs=[], execution_count=None), before='aaa1'),
    dict(op='delete', id='bbb2')]
await nb.apply(ops)
apply_ops(view, ops)
test_eq([c['id'] for c in view], [c['id'] for c in await nb.cells()])
[c['id'] for c in view]

## Change broadcasts

A kernel created with a `path` binds to that notebook, and every client on its websocket hears about changes to it: `cell_ops` messages on the `cells` queue, content `{'path', 'hash', 'ops'}`. The author of a change hears nothing, where the author is whoever's `session_id` rode the mutation. `nb` above has no `session_id`, so its writes broadcast to everyone.

In [ ]:
kc = JupyAsyncKernelClient(g.url)
await kc.start_kernel(path='d.ipynb')
kc.start_channels()
await kc.wait_for_ready(timeout=60)
kc.channels_running

In [ ]:
await nb.apply([dict(op='update', id='aaa1', source='40+2')])
m = await kc.jmsg_for('cell_ops', timeout=15)
test_eq(m['header']['msg_type'], 'cell_ops')
m['content']

The broadcast keeps a local view current through `apply_ops`, and the message's `hash` says what the file must now hash to:

In [ ]:
apply_ops(view, m['content']['ops'])
test_eq(next(c['source'] for c in view if c['id']=='aaa1'), '40+2')
test_eq(m['content']['hash'], nb.hash)
m['content']['path']

Echo suppression, live: a cells client sharing the kernel client's `session_id` counts as the same author. Its writes produce no broadcast on `kc`'s queue:

In [ ]:
own = JupyAsyncCellsClient(g.url, 'd.ipynb', session_id=kc.session_id)
await own.apply([dict(op='update', id='aaa1', source='6*7')])
try:
    await kc.jmsg_for('cell_ops', timeout=1.5)
    heard = True
except Empty: heard = False
test_eq(heard, False)

Three file-level ops can arrive in the same stream, and `apply_ops` deliberately rejects them: on `{'op':'rename','to':...}` update the client's `path`; on `{'op':'deleted'}` the file is gone; on `{'op':'reset'}` the gateway lost track; refetch with `cells()` and start over. After a reconnect that reports dropped messages, do not trust the stream either: `hashes()` against the local view, then fetch the changed cells with `ids=`.

In [ ]:
#| hide
await kc.shutdown_kernel()
await kc.aclose()
g.stop()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()